In [3]:
import cv2
import json
from ultralytics import YOLO
from tqdm import tqdm

# 1. Load the model
model = YOLO('yolov8s.pt')

# Input and output paths
input_video_path = 'demo_vid.mp4'
output_video_path = 'annotated_vid.mp4'
output_json_path = 'result.json'

cap = cv2.VideoCapture(input_video_path)

# Video properties
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0: fps = 30

# We will resize to 640x480 as requested
out_w, out_h = 640, 480

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (out_w, out_h))

# COCO to System class mapping
# COCO: 0:person, 1:bicycle, 2:car, 3:motorcycle, 5:bus, 7:truck
# System objs: 0:person, 1:bicycle, 2:car, 3:motorcycle, 4:bus, 5:truck
coco_to_objs = {
    0: (0, 'person'),
    1: (1, 'bicycle'),
    2: (2, 'car'),
    3: (3, 'motorcycle'),
    5: (4, 'bus'),
    7: (5, 'truck')
}

# COCO: 9:traffic light, 11:stop sign
# System signs: 0:traffic light, 1:stop sign
coco_to_signs = {
    9: (0, 'traffic light'),
    11: (1, 'stop sign')
}

frames_data = []

# Get total frames for the progress bar
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# 2. Use tqdm with the total frame count
pbar = tqdm(total=total_frames, desc="Processing Video")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Resize frame to 640x480
    frame = cv2.resize(frame, (out_w, out_h))
    
    # Run YOLOv8 inference
    results = model(frame, verbose=False)
    
    frame_objs = []
    frame_signs = []
    
    # Process results
    for r in results:
        boxes = r.boxes
        for box in boxes:
            cls_id = int(box.cls[0])
            xyxy = box.xyxy[0].tolist()
            xyxy = [int(x) for x in xyxy]
            
            if cls_id in coco_to_objs:
                sys_id, name = coco_to_objs[cls_id]
                frame_objs.append({
                    'class': sys_id,
                    'bbox': xyxy,
                    'name': name
                })
            elif cls_id in coco_to_signs:
                sys_id, name = coco_to_signs[cls_id]
                frame_signs.append({
                    'class': sys_id,
                    'bbox': xyxy,
                    'name': name
                })
                
    # Save frame data
    frame_data = {
        'objs': frame_objs,
        'signs': frame_signs
    }
    frames_data.append(frame_data)
    
    # Annotate frame
    annotated_frame = results[0].plot()
    out.write(annotated_frame)
    pbar.update(1)

pbar.close()
cap.release()
out.release()

# Save result.json
with open(output_json_path, 'w') as f:
    json.dump(frames_data, f, indent=4)

print(f"Processing complete. Saved to {output_video_path} and {output_json_path}")

Processing Video: 100%|██████████| 3598/3598 [01:06<00:00, 54.21it/s]


Processing complete. Saved to annotated_vid.mp4 and result.json
